In [1]:
import xarray as xr
import numpy as np
import dask
import socket
import pandas as pd
import glob
import xesmf as xe

from rossby_utils import cesm_utils as cesm
from functools import partial
import os

import importlib
importlib.reload(cesm)

import warnings
warnings.filterwarnings('ignore')

from rossby_utils import plev_interp_utils as plev_interp

### Set up the dask cluster

In [2]:
from dask_jobqueue import PBSCluster
from dask.distributed import Client
dask.config.set({"distributed.scheduler.worker_saturation":1.0})
dask.config.set({"optimization.fuse.active": False})
dask.config.set({
    "distributed.worker.memory.target": 0.6,
    "distributed.worker.memory.spill": 0.7,
    "distributed.worker.memory.pause": 0.8,
    "distributed.worker.memory.terminate": 0.95,
})

cluster = PBSCluster(
    cores = 1,
    memory = '30GB',
    processes = 1,
    queue = 'casper',
    local_directory = '/glade/derecho/scratch/islas/dask_tmp/',
    resource_spec = 'select=1:ncpus=1:mem=30GB',
    project='P04010022',
    walltime='02:00:00',
    interface='mgt')

# scale up
cluster.scale(12)
#cluster.adapt(minimum=1, maximum=12)

# change your urls to the dask dashboard so that you can see it
hostname = socket.getfqdn()
dask.config.set({"distributed.dashboard.link":
        f"https://ondemand.hpc.ucar.edu/rnode/{hostname}/{{port}}/status"
})

client = Client(cluster)

In [3]:
cluster

PBSCluster(486b872e, 'tcp://10.18.206.70:41245', workers=4, threads=4, memory=111.76 GiB)

### Surface pressure for plevel interpolation

In [8]:
phis = xr.open_dataset("/glade/campaign/cesm/collections/CESM2-LE/atm/proc/tseries/month_1/PHIS/"
     +"b.e21.BHISTcmip6.f09_g17.LE2-1001.001.cam.h0.PHIS.185001-185912.nc").isel(time=0).load()

### Set up info on data to be processed

In [4]:
ystart=1970
yend=2020 # !!! Note, SMYLE only goes to 2020
basepath="/glade/campaign/collections/gdex/data/d651083/"
outpath="/glade/campaign/cgd/cas/islas/python_savs/rossbypalooza26/processing/CESM2/SMYLE_ERA_L32/"
initmon=11
nmems=20
expname="b.e21.BSMYLE-CW3E.f09_g17"
vars=['v200']
levout=[200.]
ourvar='v'
cesmvar='V' # naming convention for CESM

os.makedirs(outpath, exist_ok=True)
for ivar in vars:
    os.makedirs(outpath+ivar, exist_ok=True)

plevout = np.array(levout)*100.

### Output grid

In [5]:
grid_out = xr.Dataset({'lat':(['lat'], np.arange(-90,90,1))}, {'lon': (['lon'], np.arange(0,360,1))})

### Grab the data, interpolate to the common 1 degree grid etc

In [6]:
def preprocessor(ds):
    # Fixing small round-off level differences in coordinates
    ds = cesm.round_lons_and_lats(ds)
    # Fix the CESM time axis
    ds = cesm.fix_cesm_time(ds)
    return ds

In [9]:
reusewgt=False
wgtfile=outpath+'wgtfile.nc'
alltimes=[]
alldat=[]
for iyear in range(ystart,yend+1,1):
    print(iyear)
    hindcast_months = pd.to_datetime([
        f"{iyear}-11-15",
        f"{iyear}-12-15",
        f"{iyear+1}-01-15",
        f"{iyear+1}-02-15",
        f"{iyear+1}-03-15"])

    # get the filelist for all November initializations for this year

    # fixing basepath for iyear = 2020.  This hasn't been moved to GDEX
      
    filelist = [ 
        glob.glob(basepath+expname+'/'+expname+'.'+str(iyear)+'-'+str(initmon).zfill(2)+'.'+str(imem).zfill(3)+'/atm/proc/tseries/month_1/*.cam.h0.'+cesmvar+'.*.nc')
        for imem in np.arange(1,nmems+1,1) ]

    filelist_ps = [ 
        glob.glob(basepath+expname+'/'+expname+'.'+str(iyear)+'-'+str(initmon).zfill(2)+'.'+str(imem).zfill(3)+'/atm/proc/tseries/month_1/*.cam.h0.PS.*.nc')
        for imem in np.arange(1,nmems+1,1) ]

    filelist_t = [ 
        glob.glob(basepath+expname+'/'+expname+'.'+str(iyear)+'-'+str(initmon).zfill(2)+'.'+str(imem).zfill(3)+'/atm/proc/tseries/month_1/*.cam.h0.T.*.nc')
        for imem in np.arange(1,nmems+1,1) ]
    
        
    dat = xr.open_mfdataset(filelist, concat_dim=['member','time'], combine='nested', preprocess = partial(preprocessor))
    hyam = dat.hyam
    hybm = dat.hybm
    dat = dat[cesmvar]
    dat = dat.chunk({"member":20})

    t = xr.open_mfdataset(filelist_t, concat_dim=['member','time'], combine='nested', preprocess = partial(preprocessor))
    t = t.T
    t = t.chunk({"member":20})
    
    ps = xr.open_mfdataset(filelist_ps, concat_dim=['member','time'], combine='nested', preprocess = partial(preprocessor))
    ps = ps.PS
    ps = ps.chunk({"member":20})

    # select out November to March
    dat = dat.sel(time=slice(str(iyear)+'-11-01',str(iyear+1)+'-03-31'))
    ps = ps.sel(time=slice(str(iyear)+'-11-01',str(iyear+1)+'-03-31'))
    t = t.sel(time=slice(str(iyear)+'-11-01',str(iyear+1)+'-03-31'))

    dat_plev =  plev_interp.interp_hybrid_to_pressure(
        dat, ps, hyam, hybm, p0=1e5, new_levels = plevout, method='log',
        lev_dim='lev', extrapolate=False, variable='other',
        t_bot = t.isel(lev=t.lev.size-1), phi_sfc = phis)

    
    # regrid
    if (iyear == ystart):
        regridder = xe.Regridder(dat_plev, grid_out, 'bilinear', periodic=True, reuse_weights=reusewgt, filename=wgtfile)
    dat_rg = regridder(dat_plev)

    # turn time into a lead axis
    dat_rg = dat_rg.rename({'time':'lead'})
    dat_rg['lead'] = np.arange(1,5+1,1)

    # comupte and append
    dat_rg = dat_rg.rename(ourvar)
    dat_rg = dat_rg.compute()

    times = xr.DataArray(hindcast_months, dims=['lead'], coords=[np.arange(1,5+1,1)], name='time')
    
    for ilev in  np.arange(0,len(levout),1):
        datout = xr.merge([dat_rg.sel(plev=levout[ilev]*100.), times])
        datout.to_netcdf(outpath+'/'+vars[ilev]+'/CESM2_SMYLE_ERA5_L32_'+vars[ilev]+'_'+str(iyear)+'.nc')

    
#    alldat.append(dat_rg)
#    
#    times = xr.DataArray(hindcast_months, dims=['lead'], coords=[np.arange(1,5+1,1)], name='time')
#    alltimes.append(times)

#alldat = xr.concat(alldat, dim='year')
#alldat['year'] = range(ystart,yend+1,1)
#alldat = alldat.rename(ourvar)
#alltimes = xr.concat(alltimes, dim='year')
#alltimes['year'] = range(ystart,yend+1,1)

#datout = xr.merge([alldat, alltimes])
#datout.to_netcdf(outpath+'CESM2_SMYLE_ERA5_L83'+ourvar+'_'+str(ystart)+'_'+str(yend)+'.nc')

1970
1971
1972
1973
1974
1975
1976
1977
1978
1979
1980
1981
1982
1983
1984
1985
1986
1987
1988
1989
1990
1991
1992
1993
1994
1995
1996
1997
1998
1999
2000
2001
2002
2003
2004
2005
2006
2007
2008
2009
2010
2011
2012
2013
2014
2015
2016
2017
2018
2019
2020


In [12]:
cluster.close()